# video-slide-md — Google Colab

Extract slide intervals from lecture videos, align subtitles, and export PPTX/Markdown presentations.

**GPU**: T4 (free) — enable via Runtime → Change runtime type → T4 GPU

---

## Инструкция
1. **Runtime → Change runtime type → T4 GPU**
2. **Run all cells** (по порядку)
3. Загрузи видео и субтитры (или укажи путь на Google Drive)
4. Результат: `slides.json`, `deck.md`, `deck.pptx`, `slides/*.png`
5. Скачай через Drive или Files panel слева

## 1. Проверка GPU

In [ ]:
import subprocess, sys, torch

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU"
print(f"GPU: {gpu_name}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "")
assert torch.cuda.is_available(), "T4 GPU not detected — set Runtime → Change runtime type → T4 GPU"

## 2. Установка зависимостей

In [ ]:
# Install system deps + video-slide-md from GitHub
!apt-get update -qq && apt-get install -y -qq ffmpeg libsm6 libxext6 2>&1 | tail -3

# Install the package with pptx and gpu extras
!pip install -q git+https://github.com/kucheryavenkovn/video2pptx.git[pptx,gpu]

print("Dependencies installed.")

## 3. Монтирование Google Drive

Загрузи видео и субтитры в папку на Drive или используй загрузчик ниже.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Укажи путь к папке с видео (или оставь по умолчанию)
DRIVE_DIR = "/content/drive/MyDrive/video_slides"
import os
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f"Drive folder: {DRIVE_DIR}")

## 4. Загрузка видео и субтитров

Загрузи файлы через Colab UI или укажи существующие на Drive.

In [ ]:
from google.colab import files
from pathlib import Path

print("Загрузи видео (.mp4):")
uploaded_video = files.upload()
video_name = list(uploaded_video.keys())[0]
print(f"\nВидео: {video_name} ({len(uploaded_video[video_name]) / 1e6:.1f} MB)")

subs_name = None
yn = input("Загрузить субтитры (.srt/.vtt)? (y/n): ").strip().lower()
if yn == 'y' or yn == 'д':
    uploaded_subs = files.upload()
    subs_name = list(uploaded_subs.keys())[0]
    print(f"Субтитры: {subs_name}")

## 5. Настройки детекции

In [ ]:
# Параметры детекции — отредактируй при необходимости
OUTPUT_DIR = "/content/out"
SAMPLE_FPS = 0.2        # кадров в секунду (0.2 = каждые 5с)
THRESHOLD = 0.05        # порог детекции смены слайда
MIN_SLIDE_DURATION = 10 # минимальная длительность слайда (сек)
MIN_STABLE_DURATION = 5 # минимальная стабильная зона (сек)
EXPORT_PPTX = True      # экспорт deck.pptx
EXPORT_MD = True        # экспорт deck.md

print(f"Output: {OUTPUT_DIR}")
print(f"Sample FPS: {SAMPLE_FPS}")
print(f"Threshold: {THRESHOLD}")
print(f"Min slide duration: {MIN_SLIDE_DURATION}s")
print(f"Min stable duration: {MIN_STABLE_DURATION}s")
print(f"Export PPTX: {EXPORT_PPTX}")
print(f"Export MD: {EXPORT_MD}")

## 6. Запуск детекции

Pipeline: frame sampling → slide change detection → segmentation → dedupe → subtitle alignment → save

In [ ]:
import sys, os, subprocess
from pathlib import Path

video_path = Path(video_name).resolve()
out_dir = Path(OUTPUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

# Build CLI args
cmd = [
    "video-slide-md", "detect",
    str(video_path),
    "--out", str(out_dir),
    "--decoder-backend", "auto",
    "--sample-fps", str(SAMPLE_FPS),
    "--threshold", str(THRESHOLD),
    "--min-slide-duration", str(MIN_SLIDE_DURATION),
    "--min-stable-duration", str(MIN_STABLE_DURATION),
    "--debug",
]

if subs_name:
    cmd.extend(["--subtitles", str(Path(subs_name).resolve())])
if EXPORT_MD:
    cmd.append("--export-md")
if EXPORT_PPTX:
    cmd.append("--export-pptx")

print("Running: " + " ".join(cmd))
print("=" * 60)

result = subprocess.run(cmd, capture_output=False)
if result.returncode != 0:
    print(f"\nERROR: exit code {result.returncode}")
    sys.exit(result.returncode)

## 7. Результаты

In [ ]:
import json

slides_json = out_dir / "slides.json"
if slides_json.exists():
    doc = json.loads(slides_json.read_text())
    print(f"Слайдов: {len(doc['slides'])}")
    print(f"Длительность видео: {doc['video']['duration']:.0f}s")
    print()
    for s in doc['slides']:
        d = s['duration']
        print(f"  #{s['index']:2d}  {_fmt(s['start'])} - {_fmt(s['end'])}  ({d:.0f}s)")
        print(f"       {s['transcript'][:100]}..." if s['transcript'] else "")

# Show exported files
print()
for f in out_dir.iterdir():
    if f.is_file():
        print(f"  {f.name}  ({f.stat().st_size / 1024:.0f} KB)")

def _fmt(sec):
    h, m, s = int(sec//3600), int((sec%3600)//60), int(sec%60)
    return f"{h}:{m:02d}:{s:02d}" if h else f"{m}:{s:02d}"

## 8. Скачивание результатов

Скопируй в Google Drive или скачай архив.

In [ ]:
# Копирование в Google Drive
import shutil

drive_dest = Path(DRIVE_DIR) / "results"
if drive_dest.exists():
    shutil.rmtree(str(drive_dest))
shutil.copytree(str(out_dir), str(drive_dest))
print(f"Results copied to: {drive_dest}")
print(f"\nDrive folder contents:")
for f in sorted(drive_dest.iterdir()):
    sz = f.stat().st_size / 1024
    print(f"  {f.name:30s} {sz:>8.0f} KB")

In [ ]:
# Скачать ZIP-архив
!zip -q -r /content/results.zip /content/out
from google.colab import files
files.download("/content/results.zip")
print("Download started.")